# Tarea 3: Transformer con Llamadas a Funciones

Sistema bilingüe que genera código Python para ejecutar funciones `relative_day()` y `next_day()` a partir de audio.

## Funciones de Fecha

In [ ]:
from datetime import datetime, timedelta

def relative_day(days_to_add, current_date='21/11/2025'):
    """Calcula fecha relativa sumando días."""
    date_format = "%d/%m/%Y"
    current = datetime.strptime(current_date, date_format)
    new_date = current + timedelta(days=int(days_to_add))
    return new_date.strftime(date_format)

def next_day(day_name, current_date='21/11/2025'):
    """Calcula la próxima ocurrencia de un día de la semana."""
    date_format = "%d/%m/%Y"
    current = datetime.strptime(current_date, date_format)
    
    # Mapeo de nombres de días a códigos ISO
    day_name = day_name.strip().lower()
    day_map = {
        "monday": 1, "lunes": 1,
        "tuesday": 2, "martes": 2,
        "wednesday": 3, "miércoles": 3, "miercoles": 3,
        "thursday": 4, "jueves": 4,
        "friday": 5, "viernes": 5,
        "saturday": 6, "sábado": 6, "sabado": 6,
        "sunday": 7, "domingo": 7,
    }
    
    if day_name not in day_map:
        raise ValueError(f"Día inválido: {day_name}")
    
    target_code = day_map[day_name]
    days_ahead = (target_code - current.isoweekday()) % 7
    if days_ahead == 0:
        days_ahead = 7  # Siempre próxima ocurrencia
    
    new_date = current + timedelta(days=days_ahead)
    return new_date.strftime(date_format)

# Probar funciones
print("Pruebas de funciones:")
print(f"relative_day(+1): {relative_day(1)}")
print(f"relative_day(+2): {relative_day(2)}")
print(f"next_day('thursday'): {next_day('thursday')}")
print(f"next_day('friday'): {next_day('friday')}")

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchaudio
import pandas as pd
import numpy as np
import random
import os
import pickle
from scipy.signal import fftconvolve
from IPython.display import Audio, display
import matplotlib.pyplot as plt

# Augmentation classes
class NoiseAug:
    """Añade ruido aditivo de MUSAN."""
    def __init__(self, noise_folder='../musan_small/noise/free-sound', 
                 snr_range=(5,15), prob=0.5, fs=16000):
        self.prob = prob
        self.snr_range = snr_range
        self.fs = fs
        if os.path.exists(noise_folder):
            self.noise_files = [os.path.join(noise_folder, f) for f in os.listdir(noise_folder) if f.endswith('.wav')]
        else:
            self.noise_files = []
    
    def __call__(self, x):
        if random.random() > self.prob or len(self.noise_files) == 0:
            return x
        
        noise_path = random.choice(self.noise_files)
        noise, _ = torchaudio.load(noise_path)
        noise = noise[0].numpy()
        
        if len(noise) < len(x):
            repeats = int(np.ceil(len(x) / len(noise)))
            noise = np.tile(noise, repeats)[:len(x)]
        else:
            start = random.randint(0, len(noise) - len(x))
            noise = noise[start:start+len(x)]
        
        snr_db = random.uniform(*self.snr_range)
        signal_power = np.mean(x ** 2)
        noise_power = np.mean(noise ** 2)
        
        if noise_power > 0:
            snr_linear = 10 ** (snr_db / 10)
            scale = np.sqrt(signal_power / (snr_linear * noise_power))
            noise = noise * scale
        
        return x + noise

class RIRAug:
    """Añade reverberación con RIR."""
    def __init__(self, rir_folder='../RIRS_NOISES_small/simulated_rirs/largeroom', 
                 prob=0.5, fs=16000):
        self.prob = prob
        self.fs = fs
        if os.path.exists(rir_folder):
            self.rir_files = [os.path.join(rir_folder, f) for f in os.listdir(rir_folder) if f.endswith('.wav')]
        else:
            self.rir_files = []
    
    def __call__(self, x):
        if random.random() > self.prob or len(self.rir_files) == 0:
            return x
        
        rir_path = random.choice(self.rir_files)
        rir, _ = torchaudio.load(rir_path)
        rir = rir[0].numpy()
        
        x_reverb = fftconvolve(x, rir, mode='same')
        
        max_val = np.max(np.abs(x_reverb))
        if max_val > 0:
            x_reverb = x_reverb / max_val * 0.9
        
        return x_reverb

# Model classes
class FeedForward(nn.Module):
    def __init__(self, d_model, d_ff, dropout=0.1):
        super().__init__()
        self.linear1 = nn.Linear(d_model, d_ff)
        self.linear2 = nn.Linear(d_ff, d_model)
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x):
        return self.linear2(self.dropout(F.gelu(self.linear1(x))))

class SelfAttention(nn.Module):
    def __init__(self, d_model, n_heads, d_head, dropout=0.1):
        super().__init__()
        self.n_heads = n_heads
        self.d_head = d_head
        
        self.qkv = nn.Linear(d_model, 3 * n_heads * d_head)
        self.out = nn.Linear(n_heads * d_head, d_model)
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x):
        B, T, C = x.shape
        q, k, v = self.qkv(x).split(self.n_heads * self.d_head, dim=-1)
        
        q = q.view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        k = k.view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        v = v.view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        
        attn = F.scaled_dot_product_attention(q, k, v, dropout_p=self.dropout.p if self.training else 0)
        
        out = attn.transpose(1, 2).contiguous().view(B, T, -1)
        return self.out(out)

class Encoder(nn.Module):
    def __init__(self, d_model, nb_layers, d_ff, n_heads, d_head, dropout=0.1):
        super().__init__()
        self.layers = nn.ModuleList([
            nn.ModuleDict({
                'attn': SelfAttention(d_model, n_heads, d_head, dropout),
                'ff': FeedForward(d_model, d_ff, dropout),
                'ln1': nn.LayerNorm(d_model),
                'ln2': nn.LayerNorm(d_model),
            }) for _ in range(nb_layers)
        ])
    
    def forward(self, x):
        for layer in self.layers:
            x = x + layer['attn'](layer['ln1'](x))
            x = x + layer['ff'](layer['ln2'](x))
        return x

class CausalSelfAttention(nn.Module):
    def __init__(self, d_model, n_heads, d_head, dropout=0.1):
        super().__init__()
        self.n_heads = n_heads
        self.d_head = d_head
        
        self.qkv = nn.Linear(d_model, 3 * n_heads * d_head)
        self.out = nn.Linear(n_heads * d_head, d_model)
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x):
        B, T, C = x.shape
        q, k, v = self.qkv(x).split(self.n_heads * self.d_head, dim=-1)
        
        q = q.view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        k = k.view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        v = v.view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        
        attn = F.scaled_dot_product_attention(
            q, k, v, 
            is_causal=True,
            dropout_p=self.dropout.p if self.training else 0
        )
        
        out = attn.transpose(1, 2).contiguous().view(B, T, -1)
        return self.out(out)

class CrossAttention(nn.Module):
    def __init__(self, d_model, n_heads, d_head, dropout=0.1):
        super().__init__()
        self.n_heads = n_heads
        self.d_head = d_head
        
        self.q = nn.Linear(d_model, n_heads * d_head)
        self.kv = nn.Linear(d_model, 2 * n_heads * d_head)
        self.out = nn.Linear(n_heads * d_head, d_model)
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x, context):
        B, T, C = x.shape
        _, T_ctx, _ = context.shape
        
        q = self.q(x).view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        k, v = self.kv(context).split(self.n_heads * self.d_head, dim=-1)
        k = k.view(B, T_ctx, self.n_heads, self.d_head).transpose(1, 2)
        v = v.view(B, T_ctx, self.n_heads, self.d_head).transpose(1, 2)
        
        attn = F.scaled_dot_product_attention(q, k, v, dropout_p=self.dropout.p if self.training else 0)
        
        out = attn.transpose(1, 2).contiguous().view(B, T, -1)
        return self.out(out)

class Decoder(nn.Module):
    def __init__(self, vocab_size, d_model, nb_layers, d_ff, n_heads, d_head, dropout=0.1, seq_len=500):
        super().__init__()
        self.tok_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Embedding(seq_len, d_model)
        
        self.layers = nn.ModuleList([
            nn.ModuleDict({
                'causal_attn': CausalSelfAttention(d_model, n_heads, d_head, dropout),
                'cross_attn': CrossAttention(d_model, n_heads, d_head, dropout),
                'ff': FeedForward(d_model, d_ff, dropout),
                'ln1': nn.LayerNorm(d_model),
                'ln2': nn.LayerNorm(d_model),
                'ln3': nn.LayerNorm(d_model),
            }) for _ in range(nb_layers)
        ])
        
        self.ln_out = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, vocab_size, bias=False)
    
    def forward(self, x, enc_out):
        B, T = x.shape
        
        tok = self.tok_emb(x)
        pos = self.pos_emb(torch.arange(T, device=x.device))
        x = tok + pos
        
        for layer in self.layers:
            x = x + layer['causal_attn'](layer['ln1'](x))
            x = x + layer['cross_attn'](layer['ln2'](x), enc_out)
            x = x + layer['ff'](layer['ln3'](x))
        
        x = self.ln_out(x)
        return self.head(x)

class SpecAug(nn.Module):
    def __init__(self, freq_mask=15, time_mask=35, n_freq=2, n_time=2):
        super().__init__()
        self.freq_mask = torchaudio.transforms.FrequencyMasking(freq_mask)
        self.time_mask = torchaudio.transforms.TimeMasking(time_mask)
        self.n_freq = n_freq
        self.n_time = n_time
    
    def forward(self, x):
        if not self.training:
            return x
        for _ in range(self.n_freq):
            x = self.freq_mask(x)
        for _ in range(self.n_time):
            x = self.time_mask(x)
        return x

class AudioFeatures(nn.Module):
    def __init__(self, feat_dim=80, n_fft=400, hop_length=160):
        super().__init__()
        self.mel = torchaudio.transforms.MelSpectrogram(
            n_mels=feat_dim, n_fft=n_fft, hop_length=hop_length
        )
    
    def forward(self, x):
        x = self.mel(x)
        x = torch.log(x + 1e-9)
        return x.transpose(1, 2)

class AudioTransformer(nn.Module):
    def __init__(self, vocab_size, d_model, nb_layers, d_ff, n_heads, d_head, 
                 dropout=0.1, seq_len=500, feat_dim=80):
        super().__init__()
        self.audio_feat = AudioFeatures(feat_dim=feat_dim)
        self.spec_aug = SpecAug()
        
        self.feat_proj = nn.Linear(feat_dim, d_model)
        self.pos_emb = nn.Embedding(seq_len, d_model)
        
        self.encoder = Encoder(d_model, nb_layers, d_ff, n_heads, d_head, dropout)
        self.decoder = Decoder(vocab_size, d_model, nb_layers, d_ff, n_heads, d_head, dropout, seq_len)
    
    def forward(self, x, y):
        x = self.audio_feat(x)
        x = self.spec_aug(x)
        x = self.feat_proj(x)
        
        B, T, C = x.shape
        pos = self.pos_emb(torch.arange(T, device=x.device))
        x = x + pos
        
        enc_out = self.encoder(x)
        logits = self.decoder(y, enc_out)
        return logits
    
    def loss(self, x, y):
        logits = self.forward(x, y[:, :-1])
        loss = F.cross_entropy(
            logits.reshape(-1, logits.size(-1)),
            y[:, 1:].reshape(-1),
            ignore_index=0
        )
        return loss
    
    @torch.no_grad()
    def generate(self, x, max_len=30, sos_token=1, eos_token=2):
        self.eval()
        
        x = self.audio_feat(x)
        x = self.feat_proj(x)
        
        B, T, C = x.shape
        pos = self.pos_emb(torch.arange(T, device=x.device))
        x = x + pos
        
        enc_out = self.encoder(x)
        
        y = torch.tensor([[sos_token]], device=x.device)
        
        for _ in range(max_len):
            logits = self.decoder(y, enc_out)
            next_token = logits[:, -1, :].argmax(dim=-1, keepdim=True)
            y = torch.cat([y, next_token], dim=1)
            
            if next_token.item() == eos_token:
                break
        
        return y[0]

print("Clases del modelo cargadas")

## Importar Librerías y Clases Necesarias

## Tokenizador con Vocabulario Extendido

In [ ]:
import pandas as pd
import torch
import pickle

class Fechas2FunctionTokenizer:
    """Tokenizador que incluye tokens para llamadas a funciones."""
    
    def __init__(self, train_es_file, train_en_file):
        # Leer archivos
        df_es = pd.read_csv(train_es_file)
        df_en = pd.read_csv(train_en_file)
        
        # Extraer vocabulario de texto y código
        words = set()
        for text in list(df_es['txt']) + list(df_en['txt']):
            # Separar texto y código por '|'
            parts = text.split('|')
            for part in parts:
                # Tokenizar conservando paréntesis, comillas, etc.
                tokens = part.strip().replace('(', ' ( ').replace(')', ' ) ').replace("'", " ' ").replace(',', ' , ').split()
                words.update([t.lower() for t in tokens])
        
        # Añadir tokens especiales para código
        special_tokens = [
            '<pad>', '<sos>', '<eos>',
            '|',  # Separador
            '(', ')', "'", ',',  # Tokens de código
            '+', '-',  # Operadores
        ]
        
        words = sorted(list(words - set(special_tokens)))
        
        self.word2index = {}
        idx = 0
        
        # Añadir tokens especiales primero
        for token in special_tokens:
            self.word2index[token] = idx
            idx += 1
        
        # Añadir vocabulario
        for word in words:
            self.word2index[word] = idx
            idx += 1
        
        self.index2word = {v: k for k, v in self.word2index.items()}
        self.vocab_size = len(self.word2index)
        
        print(f"Vocabulario con funciones: {self.vocab_size} tokens")
        print(f"Ejemplo de tokens especiales:")
        for t in ['|', '(', ')', "'", ',', '+']:
            print(f"  '{t}': {self.word2index[t]}")
    
    def encode(self, text, seq_len=-1):
        """Codifica texto con llamadas a funciones.
        
        Ejemplo: "tomorrow | relative_day(+1)" -> tokens
        """
        tokens = [self.word2index['<sos>']]
        
        # Procesar texto completo (incluyendo código)
        # Mantener '|', '(', ')', etc. como tokens separados
        text = text.replace('(', ' ( ').replace(')', ' ) ').replace("'", " ' ").replace(',', ' , ')
        
        for word in text.lower().split():
            if word in self.word2index:
                tokens.append(self.word2index[word])
            else:
                # Intentar sin espacios o tratamiento especial
                pass
        
        tokens.append(self.word2index['<eos>'])
        
        if seq_len > len(tokens):
            tokens = tokens + [self.word2index['<pad>']] * (seq_len - len(tokens))
        
        return torch.tensor(tokens)
    
    def decode(self, indices):
        """Decodifica tokens a texto."""
        if isinstance(indices, torch.Tensor):
            indices = indices.tolist()
        
        words = []
        for idx in indices:
            if idx in self.index2word:
                word = self.index2word[idx]
                if word not in ['<pad>', '<sos>', '<eos>']:
                    words.append(word)
        
        # Reconstruir texto con formato apropiado
        result = ' '.join(words)
        # Limpiar espacios alrededor de paréntesis, comillas, etc.
        result = result.replace(' ( ', '(').replace(' ) ', ')').replace(" ' ", "'")
        result = result.replace(' , ', ',').replace('( ', '(').replace(' )', ')')
        
        return result

# Crear tokenizador
tokenizer_func = Fechas2FunctionTokenizer(
    '../fechas2/fechas2_train_function.es.csv',
    '../fechas2/fechas2_train_function.en.csv'
)

# Guardar
with open('fechas2_tokenizer_function.pkl', 'wb') as f:
    pickle.dump(tokenizer_func, f)

print("\nTokenizador guardado en 'fechas2_tokenizer_function.pkl'")

### Pruebas del Tokenizador

In [ ]:
# Probar tokenizador con ejemplos de función
ejemplos = [
    "tomorrow | relative_day(+1)",
    "the day after tomorrow | relative_day(+2)",
    "next thursday | next_day('thursday')",
    "el siguiente viernes | next_day('friday')",
]

print("Pruebas del tokenizador con funciones:\n")
for text in ejemplos:
    encoded = tokenizer_func.encode(text)
    decoded = tokenizer_func.decode(encoded)
    print(f"Original:    {text}")
    print(f"Codificado:  {encoded.tolist()[:15]}...")
    print(f"Decodificado: {decoded}")
    print()

## Dataset Bilingüe con Funciones

In [ ]:
import torchaudio
import os

class Fechas2FunctionDataset(torch.utils.data.Dataset):
    """Dataset bilingüe con llamadas a funciones."""
    
    def __init__(self, train_es_csv, train_en_csv, tokenizer, 
                 audio_len=4*16000, transforms=None, max_text_len=30):
        self.tokenizer = tokenizer
        self.audio_len = audio_len
        self.transforms = transforms if transforms is not None else []
        self.max_text_len = max_text_len
        
        # Combinar ambos idiomas
        df_es = pd.read_csv(train_es_csv)
        df_en = pd.read_csv(train_en_csv)
        
        self.df = pd.concat([df_es, df_en], ignore_index=True)
        self.base_dir = os.path.dirname(train_es_csv)
        
        print(f"Dataset bilingüe con funciones: {len(self.df)} ejemplos")
        print(f"  Español: {len(df_es)}")
        print(f"  Inglés: {len(df_en)}")
    
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        
        # Cargar audio
        audio_path = os.path.join(self.base_dir, row['wav'])
        if not os.path.exists(audio_path):
            audio_path = os.path.join('..', row['wav'])
        
        x, fs = torchaudio.load(audio_path)
        
        if x.shape[1] < self.audio_len:
            x = torch.nn.functional.pad(x, (0, self.audio_len-x.shape[1]), value=0)
        else:
            x = x[:, :self.audio_len]
        
        x = x[0].numpy()
        
        # Aplicar augmentation
        for t in self.transforms:
            x = t(x)
        
        # Tokenizar (incluye texto + | + función)
        y = self.tokenizer.encode(row['txt'], seq_len=self.max_text_len)
        
        return torch.tensor(x, dtype=torch.float32), y

# Crear dataset
trainset_func = Fechas2FunctionDataset(
    '../fechas2/fechas2_train_function.es.csv',
    '../fechas2/fechas2_train_function.en.csv',
    tokenizer_func,
    transforms=[NoiseAug(prob=0.3), RIRAug(prob=0.3)]
)

## Entrenamiento

In [ ]:
# Configuración del modelo
model_config_func = {
    'vocab_size': tokenizer_func.vocab_size,
    'd_model': 256,
    'nb_layers': 6,
    'd_ff': 512,
    'n_heads': 8,
    'd_head': 32,
    'dropout': 0.1,
    'seq_len': 500,
    'feat_dim': 80
}

# Crear modelo (reutilizar AudioTransformer de tarea 1.3)
model_func = AudioTransformer(**model_config_func)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model_func.to(device)

opt = torch.optim.Adam(model_func.parameters(), lr=3e-4)

nb_epochs = 10
batch_size = 16

trainloader_func = torch.utils.data.DataLoader(
    trainset_func,
    batch_size=batch_size,
    shuffle=True
)

print(f"Iniciando entrenamiento con function calling...")

# Entrenamiento
model_func.train()
losses_func = []

for e in range(nb_epochs):
    epoch_loss = 0
    for batch_idx, (x, y) in enumerate(trainloader_func):
        x = x.to(device)
        y = y.to(device)
        
        opt.zero_grad()
        loss = model_func.loss(x, y)
        loss.backward()
        opt.step()
        
        epoch_loss += loss.item()
        
        if (batch_idx + 1) % 200 == 0:
            print(f'  Batch {batch_idx+1}/{len(trainloader_func)}: loss={loss.item():.4f}')
    
    avg_loss = epoch_loss / len(trainloader_func)
    losses_func.append(avg_loss)
    print(f'Epoch {e+1}/{nb_epochs}: avg_loss={avg_loss:.4f}')

torch.save({'model': model_func.state_dict(), 'opt': opt.state_dict(), 
            'config': model_config_func}, 'model_fechas2_function.pt')
print("Modelo guardado en 'model_fechas2_function.pt'")

## Evaluación con Ejecución de Funciones

In [ ]:
import re

class Fechas2FunctionTestDataset(torch.utils.data.Dataset):
    """Dataset de test con fechas esperadas."""
    
    def __init__(self, csv_file, tokenizer, audio_len=4*16000):
        self.df = pd.read_csv(csv_file)
        self.tokenizer = tokenizer
        self.audio_len = audio_len
        self.csv_dir = os.path.dirname(csv_file)
        print(f"Test function dataset: {len(self.df)} ejemplos")
    
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        
        audio_path = os.path.join(self.csv_dir, row['wav'])
        if not os.path.exists(audio_path):
            audio_path = os.path.join('..', row['wav'])
        
        x, fs = torchaudio.load(audio_path)
        
        if x.shape[1] < self.audio_len:
            x = torch.nn.functional.pad(x, (0, self.audio_len-x.shape[1]), value=0)
        else:
            x = x[:, :self.audio_len]
        
        return x[0], row['txt']  # txt contiene la fecha esperada

testset_func = Fechas2FunctionTestDataset(
    '../fechas2/fechas2_test_function.csv',
    tokenizer_func
)

def extract_and_execute_function(text):
    """Extrae y ejecuta la función del texto generado.
    
    Args:
        text: Texto con formato "descripción | función(args)"
    
    Returns:
        Fecha resultante en formato dd/mm/yyyy o None si hay error
    """
    try:
        # Buscar parte después de '|'
        if '|' in text:
            func_part = text.split('|')[1].strip()
        else:
            func_part = text.strip()
        
        # Ejecutar función en contexto seguro
        # Definir funciones disponibles
        safe_dict = {
            'relative_day': relative_day,
            'next_day': next_day,
        }
        
        result = eval(func_part, {"__builtins__": {}}, safe_dict)
        return result
    except Exception as e:
        print(f"Error ejecutando '{text}': {e}")
        return None

# Evaluar
model_func.eval()
correct = 0
total = 0

results_func = []

print("Evaluando modelo con function calling...\n")

for i, (x, expected_date) in enumerate(testset_func):
    x = x.to(device)
    
    # Generar predicción
    y_pred = model_func.generate(x[None,...])
    hyp = tokenizer_func.decode(y_pred)
    
    # Ejecutar función
    predicted_date = extract_and_execute_function(hyp)
    
    # Comparar
    is_correct = (predicted_date == expected_date)
    if is_correct:
        correct += 1
    total += 1
    
    results_func.append({
        'generated_text': hyp,
        'predicted_date': predicted_date,
        'expected_date': expected_date,
        'correct': is_correct
    })
    
    if i < 20:
        print(f"Ejemplo {i+1}:")
        print(f"  Generado: {hyp}")
        print(f"  Fecha predicha: {predicted_date}")
        print(f"  Fecha esperada: {expected_date}")
        print(f"  Correcto: {is_correct}")
        print()

# Calcular accuracy
accuracy = correct / total
error_rate = 1 - accuracy

print(f"\n{'='*50}")
print(f"Tasa de acierto: {accuracy:.2%}")
print(f"Tasa de error: {error_rate:.2%}")
print(f"Predicciones correctas: {correct}/{total}")
print(f"{'='*50}")

# Guardar resultados
results_func_df = pd.DataFrame(results_func)
results_func_df.to_csv('results_function.csv', index=False)
print("\nResultados guardados en 'results_function.csv'")

## Análisis de Errores

In [ ]:
# Analizar errores
results_df = pd.read_csv('results_function.csv')
errors = results_df[results_df['correct'] == False]

print(f"\nAnálisis de errores: {len(errors)} errores de {len(results_df)} ejemplos")
print(f"\nPrimeros 10 errores:")
for i, (idx, row) in enumerate(errors.head(10).iterrows()):
    print(f"\n{i+1}. Índice {idx}:")
    print(f"   Generado: {row['generated_text']}")
    print(f"   Fecha predicha: {row['predicted_date']}")
    print(f"   Fecha esperada: {row['expected_date']}")

## Ejemplos con Audio

In [ ]:
from IPython.display import Audio, display

# Mostrar algunos ejemplos con audio
print("Ejemplos con audio:\n")

for i in [0, 5, 10]:
    x, expected = testset_func[i]
    
    # Generar
    x = x.to(device)
    y_pred = model_func.generate(x[None,...])
    hyp = tokenizer_func.decode(y_pred)
    predicted = extract_and_execute_function(hyp)
    
    print(f"\n--- Ejemplo {i+1} ---")
    print(f"Texto generado: {hyp}")
    print(f"Fecha predicha: {predicted}")
    print(f"Fecha esperada: {expected}")
    print(f"Correcto: {predicted == expected}")
    
    # Audio
    audio_path = testset_func.df.iloc[i]['wav']
    if not os.path.exists(audio_path):
        audio_path = os.path.join('..', audio_path)
    display(Audio(audio_path))